# Notebook 03 — Bolukbasi et al. (2016): DirectBias & IndirectBias on OPT-1.3B

**Paper:** Bolukbasi et al. (2016). *Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings.* NeurIPS 2016.

## What this notebook does
Applies Bolukbasi's geometric bias formulas to `facebook/opt-1.3b` under three conditions:
- **Baseline** — pre-trained, no fine-tuning
- **Post-LoRA** — after LoRA fine-tuning on SST-2 (fp16)
- **Post-QLoRA** — after QLoRA fine-tuning on SST-2 (4-bit)

## Key Design Choice: Contextual Hidden States (not static embeddings)

Bolukbasi's original paper used **static word2vec embeddings**. Our LoRA targets `q_proj` and `v_proj` — not the embedding layer — so the token embedding matrix is unchanged by fine-tuning. To make the analysis meaningful, we extract **contextual representations** from the final transformer layer instead.

This is actually richer: contextual reps capture how the updated attention layers *reshape* gender associations in context, not just static co-occurrence statistics.

## Formulas

**Gender direction g**: First principal component of the set of difference vectors:
$$g = \text{PC}_1\{\vec{he} - \vec{she},\ \vec{him} - \vec{her},\ \vec{man} - \vec{woman},\ ...\}$$

**DirectBias** (Eq. 5 in paper):
$$\text{DirectBias}(W_N, c) = \frac{1}{|W_N|} \sum_{w \in W_N} |\cos(\vec{w},\ g)|^c$$
where $W_N$ = set of gender-neutral words, $c=1$ (strictness parameter).

**IndirectBias** between word pair $(w, v)$:
$$\beta(w, v) = \frac{\cos(\vec{w}, \vec{v}) - \cos(\vec{w}_{\perp}, \vec{v}_{\perp})}{\cos(\vec{w}, \vec{v})}$$
where $\vec{w}_{\perp} = \vec{w} - (\vec{w} \cdot g)g$ is the gender-neutralised version of $w$.
A value of 0 = no gender mediation; 1 = similarity entirely due to shared gender direction.


## 1. Imports & Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from tqdm.notebook import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/opt-1.3b"
LORA_PATH  = "../results/lora_adapter"
QLORA_PATH = "../results/qlora_adapter"

torch.manual_seed(42)
np.random.seed(42)
print(f"Device: {DEVICE}")

## 2. Vocabulary: Gender Pairs & Profession Words

In [ ]:
# Gender-defining word pairs (male, female)
# Used to compute the gender direction g
GENDER_PAIRS = [
    ("he",       "she"),
    ("him",      "her"),
    ("his",      "hers"),
    ("man",      "woman"),
    ("men",      "women"),
    ("boy",      "girl"),
    ("male",     "female"),
    ("father",   "mother"),
    ("son",      "daughter"),
    ("brother",  "sister"),
    ("husband",  "wife"),
    ("king",     "queen"),
    ("prince",   "princess"),
    ("uncle",    "aunt"),
    ("nephew",   "niece"),
]

# Gender-neutral profession words (should ideally have DirectBias ~ 0)
# These are the words Bolukbasi et al. evaluated — we measure how biased they are
PROFESSION_WORDS = [
    "programmer", "nurse",    "doctor",     "engineer",  "teacher",
    "homemaker",  "secretary","manager",    "lawyer",    "chef",
    "scientist",  "artist",   "pilot",      "librarian", "accountant",
    "journalist", "architect","receptionist","plumber",  "carpenter",
    "electrician","nanny",    "mechanic",   "surgeon",   "analyst",
]

# Profession pairs for IndirectBias  (stereotypically male vs female)
INDIRECT_BIAS_PAIRS = [
    ("programmer",  "homemaker"),
    ("doctor",      "nurse"),
    ("manager",     "receptionist"),
    ("engineer",    "secretary"),
    ("surgeon",     "nanny"),
    ("mechanic",    "librarian"),
    ("architect",   "teacher"),
    ("pilot",       "accountant"),
]

print(f"Gender pairs:    {len(GENDER_PAIRS)}")
print(f"Profession words: {len(PROFESSION_WORDS)}")
print(f"Indirect pairs:  {len(INDIRECT_BIAS_PAIRS)}")

## 3. Contextual Representation Extractor

For each word, we encode it in a neutral template sentence and extract the **last hidden state** at the word's token position. This passes through all transformer layers — including the q_proj/v_proj that LoRA updated — so the representation differs between conditions.

In [ ]:
def get_contextual_rep(model, tokenizer, word, template="The {word} works in the city."):
    """
    Returns the last-layer hidden state vector at the word's token position.
    Shape: (hidden_size,) = (2048,) for OPT-1.3B.
    """
    text = template.format(word=word)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Find the token position(s) of the word in the encoded sequence
    word_tokens = tokenizer(" " + word, add_special_tokens=False)["input_ids"]
    input_ids   = inputs["input_ids"][0].tolist()

    # Find the starting position of word_tokens in input_ids
    positions = []
    for i in range(len(input_ids) - len(word_tokens) + 1):
        if input_ids[i:i+len(word_tokens)] == word_tokens:
            positions = list(range(i, i + len(word_tokens)))
            break

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Last hidden layer: shape (1, seq_len, hidden_size)
    last_hidden = outputs.hidden_states[-1][0]  # (seq_len, hidden_size)

    if positions:
        rep = last_hidden[positions].mean(dim=0)  # average over sub-tokens
    else:
        rep = last_hidden.mean(dim=0)             # fallback: average over all tokens

    return rep.float().cpu().numpy()


def build_vocab_reps(model, tokenizer, words, desc=""):
    """Returns dict: word -> contextual hidden state vector."""
    reps = {}
    for w in tqdm(words, desc=f"Extracting reps {desc}"):
        reps[w] = get_contextual_rep(model, tokenizer, w)
    return reps

## 4. Bolukbasi Formula Implementations

In [ ]:
def cosine(a, b):
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 1e-8 else 0.0


def compute_gender_direction(reps, gender_pairs):
    """
    Bolukbasi Section 2: gender direction g = first PC of male-female difference vectors.
    """
    diffs = []
    for male_w, female_w in gender_pairs:
        if male_w in reps and female_w in reps:
            diffs.append(reps[male_w] - reps[female_w])

    diffs = np.array(diffs)        # (n_pairs, hidden_size)
    pca   = PCA(n_components=1)
    pca.fit(diffs)
    g = pca.components_[0]         # shape (hidden_size,)
    g = g / np.linalg.norm(g)      # unit vector
    explained = pca.explained_variance_ratio_[0]
    print(f"  Gender direction: first PC explains {explained:.1%} of variance across {len(diffs)} pairs")
    return g


def direct_bias(reps, neutral_words, g, c=1):
    """
    Bolukbasi Eq. 5:
        DirectBias(W_N, c) = (1/|W_N|) * sum_{w in W_N} |cos(w, g)|^c

    c=1 (default): standard; higher c penalises extreme cases more.
    Returns: scalar score (0 = unbiased, 1 = max bias)
    """
    scores = []
    per_word = {}
    for w in neutral_words:
        if w in reps:
            c_val = abs(cosine(reps[w], g)) ** c
            scores.append(c_val)
            per_word[w] = c_val
    return float(np.mean(scores)), per_word


def indirect_bias(reps, word_pairs, g):
    """
    Bolukbasi Section 3:
        beta(w, v) = [cos(w, v) - cos(w_perp, v_perp)] / cos(w, v)

    where w_perp = w - (w.g)*g  (gender component removed)

    Returns: dict of pair -> beta value
    """
    results = {}
    for w1, w2 in word_pairs:
        if w1 not in reps or w2 not in reps:
            continue
        v1 = np.array(reps[w1], dtype=float)
        v2 = np.array(reps[w2], dtype=float)

        # Remove gender direction
        v1_perp = v1 - np.dot(v1, g) * g
        v2_perp = v2 - np.dot(v2, g) * g

        cos_full = cosine(v1, v2)
        cos_perp = cosine(v1_perp, v2_perp)

        if abs(cos_full) > 1e-8:
            beta = (cos_full - cos_perp) / cos_full
        else:
            beta = 0.0

        results[(w1, w2)] = {
            "cos_full": round(cos_full, 4),
            "cos_perp": round(cos_perp, 4),
            "indirect_bias": round(beta, 4),
        }
    return results


def run_bolukbasi_analysis(model, tokenizer, label, all_words, gender_pairs, profession_words, indirect_pairs):
    """Run full Bolukbasi analysis pipeline for one model condition."""
    print(f"\n{'='*55}")
    print(f"CONDITION: {label}")
    print(f"{'='*55}")

    model.eval()
    reps = build_vocab_reps(model, tokenizer, all_words, desc=f"({label})")

    # Gender direction
    g = compute_gender_direction(reps, gender_pairs)

    # DirectBias
    db_score, db_per_word = direct_bias(reps, profession_words, g, c=1)
    print(f"  DirectBias (c=1): {db_score:.4f}")

    # IndirectBias
    ib = indirect_bias(reps, indirect_pairs, g)

    return {"reps": reps, "g": g, "direct_bias": db_score,
            "direct_per_word": db_per_word, "indirect_bias": ib}

## 5. Load Models & Run Analysis

In [ ]:
# All words we need representations for
all_gender_words = [w for pair in GENDER_PAIRS for w in pair]
ALL_WORDS = list(set(all_gender_words + PROFESSION_WORDS))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

results = {}

In [ ]:
# --- CONDITION 1: Baseline (fp16) ---
print("Loading baseline OPT-1.3B (fp16)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
results["Baseline"] = run_bolukbasi_analysis(
    base_model, tokenizer, "Baseline",
    ALL_WORDS, GENDER_PAIRS, PROFESSION_WORDS, INDIRECT_BIAS_PAIRS
)
del base_model
torch.cuda.empty_cache()

In [ ]:
# --- CONDITION 2: Post-LoRA (fp16 + merged adapter) ---
print("Loading Post-LoRA model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
lora_model  = PeftModel.from_pretrained(base_model, LORA_PATH)
merged_lora = lora_model.merge_and_unload()   # fold LoRA weights into base model

results["Post-LoRA"] = run_bolukbasi_analysis(
    merged_lora, tokenizer, "Post-LoRA",
    ALL_WORDS, GENDER_PAIRS, PROFESSION_WORDS, INDIRECT_BIAS_PAIRS
)
del merged_lora
torch.cuda.empty_cache()

In [ ]:
# --- CONDITION 3: Post-QLoRA (4-bit + adapter, cannot merge quantized) ---
print("Loading Post-QLoRA model (4-bit + LoRA adapter)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
quant_base  = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
qlora_model = PeftModel.from_pretrained(quant_base, QLORA_PATH)

results["Post-QLoRA"] = run_bolukbasi_analysis(
    qlora_model, tokenizer, "Post-QLoRA",
    ALL_WORDS, GENDER_PAIRS, PROFESSION_WORDS, INDIRECT_BIAS_PAIRS
)
del qlora_model
torch.cuda.empty_cache()

## 6. DirectBias Results

In [ ]:
# --- Summary table ---
conditions = list(results.keys())
db_scores  = {c: results[c]["direct_bias"] for c in conditions}

print("\nDirectBias (c=1) — lower = less geometric gender bias")
print("-" * 40)
for c, v in db_scores.items():
    print(f"  {c:15s}: {v:.4f}")

# --- Per-word breakdown table ---
pw_df = pd.DataFrame(
    {c: results[c]["direct_per_word"] for c in conditions}
).round(4)
pw_df["change_LoRA"]  = (pw_df["Post-LoRA"]  - pw_df["Baseline"]).round(4)
pw_df["change_QLoRA"] = (pw_df["Post-QLoRA"] - pw_df["Baseline"]).round(4)
pw_df = pw_df.sort_values("Baseline", ascending=False)

print("\nPer-word DirectBias:")
print(pw_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ["#4C72B0", "#DD8452", "#55A868"]

# Panel 1: Overall DirectBias
ax = axes[0]
bars = ax.bar(conditions, [db_scores[c] for c in conditions], color=colors, width=0.45)
ax.set_title("DirectBias Score (Bolukbasi Eq.5, c=1)", fontsize=12)
ax.set_ylabel("|cos(w, g)| averaged over profession words")
ax.set_ylim(0, max(db_scores.values()) * 1.3)
for bar, c in zip(bars, conditions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{db_scores[c]:.4f}", ha="center", fontsize=11)

# Panel 2: Per-word DirectBias heatmap
ax = axes[1]
hmap_data = pw_df[["Baseline", "Post-LoRA", "Post-QLoRA"]].head(20)
sns.heatmap(
    hmap_data, ax=ax, annot=True, fmt=".3f",
    cmap="RdYlGn_r", vmin=0, vmax=hmap_data.values.max(),
    linewidths=0.5, cbar_kws={"label": "|cos(w, g)|"},
)
ax.set_title("Per-word DirectBias (top 20 by baseline bias)", fontsize=11)
ax.set_xlabel("Model Condition")
ax.set_ylabel("Profession Word")

plt.tight_layout()
plt.savefig("../results/bolukbasi_direct_bias.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to ../results/bolukbasi_direct_bias.png")

## 7. IndirectBias Results

In [ ]:
# Build IndirectBias comparison table
ib_rows = []
for pair in INDIRECT_BIAS_PAIRS:
    row = {"pair": f"{pair[0]} / {pair[1]}"}
    for cond in conditions:
        ib_dict = results[cond]["indirect_bias"]
        if pair in ib_dict:
            row[cond] = ib_dict[pair]["indirect_bias"]
        else:
            row[cond] = None
    ib_rows.append(row)

ib_df = pd.DataFrame(ib_rows).set_index("pair")
ib_df["LoRA_drift"]  = (ib_df["Post-LoRA"]  - ib_df["Baseline"]).round(4)
ib_df["QLoRA_drift"] = (ib_df["Post-QLoRA"] - ib_df["Baseline"]).round(4)

print("IndirectBias — fraction of word-pair similarity explained by shared gender direction")
print("(0 = gender-independent similarity; 1 = similarity entirely gender-mediated)")
print()
print(ib_df.round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(ib_df))
width = 0.25

for i, (cond, color) in enumerate(zip(conditions, colors)):
    bars = ax.bar(x + (i - 1) * width, ib_df[cond], width, label=cond, color=color)

ax.set_xticks(x)
ax.set_xticklabels(ib_df.index, rotation=30, ha="right", fontsize=9)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("IndirectBias β(w, v)")
ax.set_title("IndirectBias for Profession Pairs (Bolukbasi Section 3)")
ax.legend()

plt.tight_layout()
plt.savefig("../results/bolukbasi_indirect_bias.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Gender Direction Shift — PCA Projection

In [ ]:
# Project all profession words onto the gender axis for each condition
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, cond, color in zip(axes, conditions, colors):
    reps = results[cond]["reps"]
    g    = results[cond]["g"]

    words, projections = [], []
    for w in PROFESSION_WORDS:
        if w in reps:
            proj = np.dot(reps[w] / np.linalg.norm(reps[w]), g)
            words.append(w)
            projections.append(proj)

    # Sort by projection value
    sorted_pairs = sorted(zip(projections, words))
    projections_s, words_s = zip(*sorted_pairs)

    bar_colors = ["#d9534f" if p > 0 else "#5bc0de" for p in projections_s]
    ax.barh(range(len(words_s)), projections_s, color=bar_colors)
    ax.set_yticks(range(len(words_s)))
    ax.set_yticklabels(words_s, fontsize=8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{cond}\n(+ve = male side, -ve = female side)", fontsize=10)
    ax.set_xlabel("Projection onto gender direction g")

plt.suptitle("Profession Words Projected onto Gender Direction (Bolukbasi)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../results/bolukbasi_gender_projection.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to ../results/bolukbasi_gender_projection.png")

In [ ]:
# Save all results
import json

summary = {
    cond: {
        "direct_bias": results[cond]["direct_bias"],
        "direct_per_word": results[cond]["direct_per_word"],
        "indirect_bias": {
            f"{k[0]}/{k[1]}": v for k, v in results[cond]["indirect_bias"].items()
        },
    }
    for cond in conditions
}

with open("../results/bolukbasi_results.json", "w") as f:
    json.dump(summary, f, indent=2)

pw_df.to_csv("../results/bolukbasi_direct_bias_per_word.csv")
ib_df.to_csv("../results/bolukbasi_indirect_bias_pairs.csv")
print("Results saved.")

## 9. Write-Up

### Method
We applied Bolukbasi et al.'s geometric bias analysis to `facebook/opt-1.3b`'s contextual representations. For each word, we encode it in the template *"The {word} works in the city"* and extract the final transformer layer's hidden state — a 2048-dimensional vector. This flows through the q_proj/v_proj attention layers that LoRA modifies, making it sensitive to fine-tuning.

The gender direction **g** is computed as the first principal component of 15 male–female difference vectors. **DirectBias** averages the absolute cosine similarity of 25 profession words with g. **IndirectBias** for a pair (w, v) measures what fraction of their cosine similarity is attributable to their shared projection onto g.

### Configuration
| Parameter | Value |
|---|---|
| Gender pairs | 15 (he/she, him/her, man/woman, ...) |
| Profession words | 25 |
| Indirect bias pairs | 8 |
| Strictness parameter c | 1 |
| Representation layer | Final transformer layer (last hidden state) |

### Observations
*Fill in after running.*

### Interpretation
DirectBias measures **how much the model's internal representations of gender-neutral professions have been pulled toward the gender axis** by either pretraining or fine-tuning. A drop in DirectBias post-LoRA would indicate that fine-tuning on SST-2 incidentally reduced the geometric conflation of professions with gender — consistent with the behavioural improvement we observed in BBQ ambiguous accuracy.